# Decision Trees (CART) — learning a flowchart

> Tutorial pair for [`decision_tree.py`](decision_tree.py).

## 1. Intuition
A decision tree is a flowchart of yes/no questions on the features. Each question
splits the data to make the resulting groups **purer** (more single-class). Keep
splitting until groups are pure enough, then predict the majority class (or mean
value) in each leaf. Interpretable, nonlinear, scale-invariant.

## 2. Concept (the slide)
- **Greedy growth:** at each node pick the (feature, threshold) that most reduces
  impurity.
- **Impurity:** Gini or entropy (classification), variance/MSE (regression).
- **Pre-pruning:** cap `max_depth` / require `min_samples_split` to avoid
  memorizing noise (a fully grown tree overfits).
- Base learner for **random forests** (bagging) and **gradient boosting**.

## 3. Math derivation

**Impurity measures** for a node with class proportions $p_c$:

$$\text{Gini}=1-\sum_c p_c^2,\qquad
  \text{Entropy}=-\sum_c p_c\log_2 p_c,\qquad
  \text{(regression) } \text{MSE}=\tfrac1{|S|}\sum_{i\in S}(y_i-\bar y)^2 .$$

**Best split.** Splitting set $S$ into $S_L,S_R$ has weighted child impurity

$$I_{\text{split}}=\frac{|S_L|}{|S|}\,I(S_L)+\frac{|S_R|}{|S|}\,I(S_R),$$

and the **information gain** is $\Delta=I(S)-I_{\text{split}}$. CART scans every
feature and every candidate threshold (midpoints between sorted unique values)
and picks the split with the largest $\Delta$ (= smallest $I_{\text{split}}$).
This is repeated recursively — a greedy, locally optimal procedure.

**Why Gini vs entropy?** Both peak at a uniform class mix and vanish at purity;
Gini avoids a logarithm (slightly cheaper) and usually gives near-identical
trees. Entropy/information gain is the ID3/C4.5 lineage.

**Overfitting & pruning.** An unconstrained tree can place every point in its own
leaf (zero training error, terrible generalization — pure **variance**).
Pre-pruning (`max_depth`, `min_samples_split`, `min_impurity_decrease`) or
post-pruning (cost-complexity $R_\alpha(T)=R(T)+\alpha|T|$) trades a little bias
for much less variance. You'll see the test accuracy peak at moderate depth.

## 4. NumPy implementation (CART: classification + regression + pruning)

In [ ]:
# ===== actual implementation from decision_tree.py =====
from __future__ import annotations

import numpy as np

SEED = 0

def _gini(y):
    _, c = np.unique(y, return_counts=True); p = c / c.sum()
    return 1.0 - (p ** 2).sum()

def _entropy(y):
    _, c = np.unique(y, return_counts=True); p = c / c.sum()
    return -(p * np.log2(p + 1e-12)).sum()

def _mse(y):
    return float(np.var(y)) if len(y) else 0.0

class _Node:
    __slots__ = ("feature", "threshold", "left", "right", "value")

    def __init__(self):
        self.feature = self.threshold = self.left = self.right = self.value = None

def demo():
    np.random.seed(SEED)
    from sklearn.datasets import load_iris, make_friedman1

    # classification
    X, y = load_iris(return_X_y=True)
    idx = np.random.permutation(len(X)); X, y = X[idx], y[idx]
    Xtr, ytr, Xte, yte = X[:120], y[:120], X[120:], y[120:]
    for depth in (1, 3, None):
        t = DecisionTreeNumPy(max_depth=depth).fit(Xtr, ytr)
        print(f"[clf] max_depth={str(depth):>4}  depth={t.depth()}  "
              f"test acc={np.mean(t.predict(Xte) == yte):.3f}")

    sk = sklearn_reference(Xtr, ytr, max_depth=3)
    print(f"[clf] sklearn (depth 3) acc={np.mean(sk.predict(Xte) == yte):.3f}")

    # regression
    Xr, yr = make_friedman1(n_samples=300, noise=1.0, random_state=SEED)
    tr = DecisionTreeNumPy(task="regression", max_depth=4).fit(Xr[:240], yr[:240])
    mse = np.mean((tr.predict(Xr[240:]) - yr[240:]) ** 2)
    print(f"[reg] test MSE (depth 4): {mse:.3f}")


class DecisionTreeNumPy:
    def __init__(self, task="classification", criterion=None,
                 max_depth=None, min_samples_split=2):
        self.task = task
        self.criterion = criterion or ("gini" if task == "classification" else "mse")
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self._imp = {"gini": _gini, "entropy": _entropy, "mse": _mse}[self.criterion]

    # weighted impurity after a split (the thing we minimize)
    def _split_score(self, y, mask):
        l, r = y[mask], y[~mask]
        if len(l) == 0 or len(r) == 0:
            return np.inf
        n = len(y)
        return (len(l) * self._imp(l) + len(r) * self._imp(r)) / n

    def _best_split(self, X, y):
        best = (np.inf, None, None)
        for f in range(X.shape[1]):
            thresholds = np.unique(X[:, f])
            # midpoints between consecutive unique values
            for t in (thresholds[:-1] + thresholds[1:]) / 2 if len(thresholds) > 1 else []:
                mask = X[:, f] <= t
                score = self._split_score(y, mask)
                if score < best[0]:
                    best = (score, f, t)
        return best  # (score, feature, threshold)

    def _leaf_value(self, y):
        if self.task == "classification":
            vals, c = np.unique(y, return_counts=True)
            return vals[c.argmax()]
        return float(np.mean(y))

    def _build(self, X, y, depth):
        node = _Node()
        # stopping rules (pre-pruning)
        if (len(y) < self.min_samples_split or
                (self.max_depth is not None and depth >= self.max_depth) or
                len(np.unique(y)) == 1):
            node.value = self._leaf_value(y); return node
        score, f, t = self._best_split(X, y)
        if f is None or score == np.inf:
            node.value = self._leaf_value(y); return node
        mask = X[:, f] <= t
        node.feature, node.threshold = f, t
        node.left = self._build(X[mask], y[mask], depth + 1)
        node.right = self._build(X[~mask], y[~mask], depth + 1)
        return node

    def fit(self, X, y):
        self.root = self._build(np.asarray(X, float), np.asarray(y), 0)
        return self

    def _predict_one(self, x, node):
        if node.value is not None:
            return node.value
        branch = node.left if x[node.feature] <= node.threshold else node.right
        return self._predict_one(x, branch)

    def predict(self, X):
        return np.array([self._predict_one(x, self.root) for x in np.asarray(X, float)])

    def depth(self, node=None):
        node = node or self.root
        if node.value is not None:
            return 0
        return 1 + max(self.depth(node.left), self.depth(node.right))

## 5. "PyTorch" note

Trees are discrete, non-differentiable greedy structures — there is no gradient to
backprop, so PyTorch isn't the natural tool (it shines for differentiable models;
*soft/differentiable* trees exist but are a research variant). We instead
cross-check against scikit-learn's optimized CART.

In [ ]:
# ===== actual implementation from decision_tree.py =====
def sklearn_reference(X, y, task="classification", **kw):
    from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
    Tree = DecisionTreeClassifier if task == "classification" else DecisionTreeRegressor
    return Tree(random_state=SEED, **kw).fit(X, y)

## 6. Train — depth vs accuracy, and the sklearn cross-check

In [ ]:
demo()

## 7. Visualization — decision regions get more complex with depth

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from sklearn.datasets import make_moons
import decision_tree as M

X, y = make_moons(n_samples=300, noise=0.25, random_state=0)
xx, yy = np.meshgrid(np.linspace(X[:,0].min()-.5, X[:,0].max()+.5, 200),
                     np.linspace(X[:,1].min()-.5, X[:,1].max()+.5, 200))
grid = np.c_[xx.ravel(), yy.ravel()]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, depth in zip(axes, (1, 3, 8)):
    t = M.DecisionTreeNumPy(max_depth=depth).fit(X, y)
    zz = t.predict(grid).reshape(xx.shape)
    ax.contourf(xx, yy, zz, alpha=.3, cmap="coolwarm")
    ax.scatter(X[:,0], X[:,1], c=y, s=12, edgecolor="k", cmap="coolwarm")
    ax.set_title(f"max_depth = {depth}")   # axis-aligned, staircase boundaries
plt.tight_layout(); plt.show()

## 8. Takeaways & pitfalls
- Boundaries are **axis-aligned staircases**; deep trees overfit (high variance).
- No feature scaling needed; handles mixed feature types and is interpretable.
- A single tree is weak/unstable → **ensembles**: random forest (bag many trees
  on bootstrap samples + feature subsets) and gradient boosting (fit trees to
  residuals). Those are the next files in `ml/trees/` and `ml/ensemble/`.